# CognitiveRec: Group E

Runs the four project stages. All code lives in `src/cognitiverec/`, so this notebook stays small and doesn't cause merge conflicts: **change the code in the repo, not here, and don't commit outputs.**

Works on Google Colab (CPU runtime) and locally, with nothing extra to install. Stages 1 and 2 re-tune and overwrite `configs/tuned/` in this copy of the repo.


In [ ]:
# Setup. Owner: all.
import os
import subprocess
import sys
from pathlib import Path

# Must be set before NumPy loads: extra BLAS threads cost CPU here without saving time.
for variable in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[variable] = "1"

if "google.colab" in sys.modules:
    root = Path("/content/CognitivRec")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/jobz3/CognitivRec.git", str(root)], check=True
        )
else:
    root = Path.cwd().resolve()
    while not (root / "pyproject.toml").exists():
        root = root.parent

# A running kernel may not see an editable install until it restarts, so use sys.path instead.
sys.path.insert(0, str(root / "src"))

from IPython.display import Image, display

from cognitiverec.config import load_config
from cognitiverec.experiments import (
    stage1_baselines,
    stage2_cognitive,
    stage3_trust,
    stage4_ablation,
)

# True runs stages 3 and 4 on one seed, as a quick check. Use False for numbers in the report.
QUICK = False
evaluation_overrides = {"seeds": [42]} if QUICK else None

## Stage 1: data and baselines

Owners: Jobin (Models A and B), Mudit (data pipeline). Tunes on validation, then reports on test.

In [ ]:
stage1 = stage1_baselines.run()

## Stage 2: Models C and D

Owners: Mudit (Model C), Abhinav (Model D). Tunes on validation only.

In [ ]:
stage2 = stage2_cognitive.run()

## Stage 3: trust evaluation

Owners: all. RQ1 to RQ4 for every model on the test split, using the tuned hyperparameters.

In [ ]:
stage3 = stage3_trust.run(load_config("stage3_trust", evaluation_overrides))
for figure in stage3.figures:
    display(Image(filename=str(figure), width=800))

## Stage 4: ablations and trade-offs

Owners: Mudit (ablations), Abhinav (trade-off figures).

In [ ]:
stage4 = stage4_ablation.run(load_config("stage4_ablation", evaluation_overrides))
for figure in stage4.figures:
    display(Image(filename=str(figure), width=640))